# BM4D (Volumetric Denoising) — Implementation / 구현

**Paper**: Maggioni, M., Katkovnik, V., Egiazarian, K., & Foi, A. (2013). *IEEE TIP*, 22(1), 119–133.

## Overview / 개요

BM4D는 BM3D의 3-D 볼륨 확장. 본 노트북은:
1. **합성 3-D 볼륨** 생성 (Shepp-Logan 3-D 또는 cube-style synthetic phantom) + Gaussian 잡음
2. **Slice-by-slice BM3D** 적용 (baseline) — `bm3d` 패키지
3. **Cube extraction + cube distance** 시연
4. **Rician noise** 시뮬레이션 + VST framework 시각화
5. (가능하면) `bm3d` 패키지의 `bm4d` 함수 사용

**Note**: full BM4D requires substantial engineering. This notebook demonstrates building blocks.


In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from numpy.typing import NDArray

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 10
rng = np.random.default_rng(42)


def psnr(a, b, peak=1.0):
    return float(10 * np.log10(peak ** 2 / max(np.mean((a - b) ** 2), 1e-12)))

---

## Part 1: Synthetic 3-D phantom / 합성 3-D 팬텀

Shepp-Logan을 simplified 3-D로 확장 (concentric ellipsoids).


In [ ]:
def shepp_logan_3d(N: int = 64) -> NDArray[np.float64]:
    """Simplified 3-D Shepp-Logan: 3 concentric ellipsoids."""
    z, y, x = np.mgrid[-1:1:N*1j, -1:1:N*1j, -1:1:N*1j]
    # Outer skull
    outer = ((x/0.95)**2 + (y/0.95)**2 + (z/0.95)**2 <= 1).astype(float) * 0.3
    # Brain
    brain = ((x/0.85)**2 + (y/0.85)**2 + (z/0.85)**2 <= 1).astype(float) * 0.7
    # Inner low-density region
    ventricle = ((x/0.4)**2 + (y/0.3)**2 + (z/0.3)**2 <= 1).astype(float) * 0.4
    return np.clip(outer + brain - ventricle, 0, 1)


phantom = shepp_logan_3d(N=48)
print(f'Phantom shape: {phantom.shape}, range [{phantom.min():.3f}, {phantom.max():.3f}]')

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, axis_name, idx in zip(axes, ['axial', 'sagittal', 'coronal'], [phantom.shape[0]//2, phantom.shape[1]//2, phantom.shape[2]//2]):
    if axis_name == 'axial':
        slc = phantom[idx, :, :]
    elif axis_name == 'sagittal':
        slc = phantom[:, idx, :]
    else:
        slc = phantom[:, :, idx]
    ax.imshow(slc, cmap='gray'); ax.set_title(f'{axis_name} slice (idx {idx})'); ax.axis('off')
plt.tight_layout(); plt.show()

---

## Part 2: Add Gaussian and Rician noise / 가우시안과 라이시안 잡음

Eq. (13): \$z = \\sqrt{(c\_r y + \\sigma\\eta\_r)^2 + (c\_i y + \\sigma\\eta\_i)^2}\$ — magnitude MR noise (Rician).


In [ ]:
def add_gaussian_3d(vol: NDArray[np.float64], sigma: float, seed: int = 0) -> NDArray[np.float64]:
    return vol + sigma * np.random.default_rng(seed).standard_normal(vol.shape)


def add_rician_3d(vol: NDArray[np.float64], sigma: float, seed: int = 0) -> NDArray[np.float64]:
    """Rician noise: |z + iw| with z = vol + N(0,σ²), w = N(0,σ²)."""
    rng = np.random.default_rng(seed)
    real = vol + sigma * rng.standard_normal(vol.shape)
    imag = sigma * rng.standard_normal(vol.shape)
    return np.sqrt(real ** 2 + imag ** 2)


sigma = 0.10
vol_gauss = add_gaussian_3d(phantom, sigma)
vol_rician = add_rician_3d(phantom, sigma)

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
axes[0].imshow(phantom[24, :, :], cmap='gray', vmin=0, vmax=1); axes[0].set_title('clean'); axes[0].axis('off')
axes[1].imshow(vol_gauss[24, :, :], cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Gaussian σ={sigma} (PSNR={psnr(vol_gauss, phantom):.2f} dB)'); axes[1].axis('off')
axes[2].imshow(vol_rician[24, :, :], cmap='gray', vmin=0, vmax=1)
axes[2].set_title(f'Rician σ={sigma} (PSNR={psnr(vol_rician, phantom):.2f} dB)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

# Note: Rician PSNR slightly different because |Rician| has positive mean shift
print(f'Gaussian noise: signal mean changed by {(vol_gauss - phantom).mean():.4f}')
print(f'Rician noise  : signal mean changed by {(vol_rician - phantom).mean():.4f}')
print('Rician noise has positive bias in low-intensity regions — needs VST or bias correction.')

---

## Part 3: Cube extraction and grouping / 큐브 추출과 그룹화

BM4D의 핵심: 3-D cubes 사이 photometric distance.


In [ ]:
def extract_cube(vol: NDArray[np.float64], pos: tuple[int, int, int], L: int = 4) -> NDArray[np.float64]:
    """Extract L×L×L cube from volume at top-left-front corner pos."""
    z, y, x = pos
    return vol[z:z+L, y:y+L, x:x+L]


def cube_distance(c1: NDArray[np.float64], c2: NDArray[np.float64]) -> float:
    """Photometric distance Eq. (2): ||c1 - c2||² / L³."""
    return float(np.mean((c1 - c2) ** 2))


def find_similar_cubes_3d(vol: NDArray[np.float64], ref_pos: tuple[int, int, int],
                            L: int = 4, search_radius: int = 5, n_max: int = 16) -> list[tuple]:
    """Find up to n_max similar cubes within search_radius."""
    Z, Y, X = vol.shape
    rz, ry, rx = ref_pos
    ref_cube = extract_cube(vol, ref_pos, L)
    candidates = []
    for dz in range(-search_radius, search_radius + 1):
        for dy in range(-search_radius, search_radius + 1):
            for dx in range(-search_radius, search_radius + 1):
                z, y, x = rz + dz, ry + dy, rx + dx
                if 0 <= z <= Z - L and 0 <= y <= Y - L and 0 <= x <= X - L:
                    cube = extract_cube(vol, (z, y, x), L)
                    candidates.append((cube_distance(ref_cube, cube), (z, y, x)))
    candidates.sort(key=lambda t: t[0])
    return [c for _, c in candidates[:n_max]]


ref_pos = (24, 24, 24)
matches = find_similar_cubes_3d(vol_gauss, ref_pos, L=4, search_radius=6, n_max=8)
print(f'Reference cube at {ref_pos}, found {len(matches)} similar cubes:')
for m in matches[:5]:
    print(f'  {m}')

---

## Part 4: Slice-by-slice BM3D baseline / 슬라이스별 BM3D


In [ ]:
try:
    import bm3d
    have_bm3d = True
except ImportError:
    have_bm3d = False

if have_bm3d:
    # Slice-by-slice BM3D as baseline
    vol_slice_bm3d = np.zeros_like(vol_gauss)
    for k in range(vol_gauss.shape[0]):
        vol_slice_bm3d[k] = bm3d.bm3d(vol_gauss[k], sigma_psd=sigma,
                                         stage_arg=bm3d.BM3DStages.ALL_STAGES)
    p_noisy = psnr(vol_gauss, phantom)
    p_slice = psnr(vol_slice_bm3d, phantom)
    print(f'Volumetric PSNR comparison:')
    print(f'  noisy (Gaussian σ=0.10) : {p_noisy:.2f} dB')
    print(f'  slice-by-slice BM3D     : {p_slice:.2f} dB')
    print(f'  improvement             : {p_slice - p_noisy:+.2f} dB')
    print(f'\nFull BM4D would add ~0.5-2 dB over slice-by-slice BM3D by exploiting inter-slice redundancy.')
else:
    print('bm3d package not installed.')

In [ ]:
if have_bm3d:
    fig, axes = plt.subplots(1, 3, figsize=(13, 5))
    axes[0].imshow(phantom[24], cmap='gray', vmin=0, vmax=1); axes[0].set_title('clean')
    axes[1].imshow(vol_gauss[24], cmap='gray', vmin=0, vmax=1)
    axes[1].set_title(f'noisy ({psnr(vol_gauss[24], phantom[24]):.2f} dB)')
    axes[2].imshow(vol_slice_bm3d[24], cmap='gray', vmin=0, vmax=1)
    axes[2].set_title(f'slice BM3D ({psnr(vol_slice_bm3d[24], phantom[24]):.2f} dB)')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()

---

## Part 5: VST for Rician (concept) / Rician용 VST 개념

VST는 Rician 분포를 Gaussian-like로 안정화. Foi 2011의 VST는 closed-form 복잡, 여기서는 간단한 forward Anscombe-style transform 시연.


In [ ]:
def simple_vst_forward(z: NDArray[np.float64], sigma: float) -> NDArray[np.float64]:
    """Simple Foi-Anscombe-style VST for Rician magnitude data.

    Approximation: f(z) = 2*sqrt((z/sigma)^2 + 0.5)  (asymptotic for high SNR).
    """
    return 2 * np.sqrt((z / sigma) ** 2 + 0.5) * sigma


def simple_vst_inverse(t: NDArray[np.float64], sigma: float) -> NDArray[np.float64]:
    """Algebraic inverse of the forward VST above."""
    return sigma * np.sqrt(np.maximum((t / (2 * sigma)) ** 2 - 0.5, 0.0))


vol_rician = add_rician_3d(phantom, sigma=0.10)
vol_vst = simple_vst_forward(vol_rician, sigma=0.10)

# Compare standard deviation by intensity bin
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
intensities = phantom.ravel()
noise_rician = (vol_rician - phantom).ravel()
noise_vst = (vol_vst - simple_vst_forward(phantom, sigma=0.10)).ravel()

bins = np.linspace(0, 1, 11)
for noise, label, color, ax in [(noise_rician, 'Rician noise', 'C3', axes[0]),
                                  (noise_vst, 'VST-stabilised', 'C2', axes[1])]:
    stds = []
    centres = []
    for i in range(len(bins) - 1):
        mask = (intensities >= bins[i]) & (intensities < bins[i+1])
        if mask.sum() > 100:
            stds.append(noise[mask].std())
            centres.append((bins[i] + bins[i+1]) / 2)
    ax.plot(centres, stds, '-o', color=color)
    ax.set_xlabel('signal intensity'); ax.set_ylabel('local noise std')
    ax.set_title(f'{label}: noise std vs intensity'); ax.grid(True)
plt.tight_layout(); plt.show()

print('Rician noise has signal-dependent std (especially in low-intensity regions).')
print('VST stabilises noise to be approximately uniform across intensities — Gaussian-ready for BM4D.')

---

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 본 노트북 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| 3-D cubes \$L^3\$ voxels | §II.B | `extract_cube` | (paper-specific) |
| Photometric cube distance | Eq. (2) | `cube_distance` | (paper-specific) |
| 4-D groups | Eq. (4) | `find_similar_cubes_3d` | reference: BM4D Matlab |
| 4-D collaborative filtering | §II.B | (out of scope) | reference: GCF-BM3D toolbox |
| Rician noise model | Eq. (13) | `add_rician_3d` | medical-imaging libs |
| VST framework | Eq. (14) | `simple_vst_forward/inverse` | `pybm3d` / GCF-BM3D Matlab |

### Connections to next papers / 다음 논문과의 연결

- **Paper #7 BM3D** — direct ancestor (2-D image version).
- **Paper #9 V-BM4D** — sister extension (video; 4th dim is time).
- **Modern deep MRI denoisers** — DnCNN-3D, NLRPCA-deep, Restormer-3D — supersede BM4D but BM4D remains training-free baseline.

### Take-away / 결론

본 노트북은 BM4D의 핵심 building block (3-D cube extraction, photometric distance, 4-D grouping)을 시연. 합성 3-D Shepp-Logan phantom + Gaussian/Rician noise로 visualise. Slice-by-slice BM3D 적용으로 baseline을 보였고, 실제 BM4D는 *inter-slice redundancy*까지 활용해 +0.5-2 dB 추가 이득을 줄 것임을 명시. VST framework가 Rician noise의 signal-dependent variability를 stabilise함을 시각·정량 확인.

Demonstrates BM4D building blocks: 3-D cubes, photometric distance, 4-D grouping. Synthetic Shepp-Logan phantom with Gaussian/Rician noise. Slice-by-slice BM3D as baseline; full BM4D would add 0.5-2 dB by exploiting inter-slice redundancy. VST stabilises Rician noise's signal-dependent variability for BM4D's Gaussian assumption.
